# V2.1 — Guarded, Historically-Contacted Walk-Forward Readout

Route: `lst_models` V2.1 extension
Scope: `guarded_walkforward_readout` (holdout contacted under guarded designation; zero official-validation scoring events)
Designation: **guarded, historically-contacted walk-forward readout**
User-facing notebook: `notebooks/v2_1_guarded_walkforward_readout_colab.ipynb`

This notebook executes the pre-registered V2.1 walk-forward readout on the post-2017-01-25 closed segment. The holdout was contacted during V1 development; all results carry the guarded qualification per `docs/protocols/v2_1_guarded_walkforward_readout_protocol.md`.

FORBIDDEN wording in any output: clean test, clean holdout, untouched test, untouched holdout, final evidence, out-of-sample proof.

## Protocol Summary

- **Walk-forward design**: k=7 expanding-window periods, each ~12 months, starting at 2017-01-25. Period 7 truncated at data end (2024-04-19).
- **Model roster**: 5 models (Dummy, LightGBM, Std DLinear, TCN, MS-DLinear+TCN), all frozen params from Stage 02/03. Zero new HPO.
- **Refit recipe**: frozen D1/D2 mechanism — chronological-tail early stopping carved from refit rows only; scored period rows never participate.
- **Seeds**: `[101, 202]` (frozen V2 seed policy).
- **Stability criteria** (judged on `tcn_frozen_primary` only): `positive_period_count >= 2` AND `pooled_delta > 0`.
- **Total scoring events**: 7 periods x 4 model rows x 2 seeds = 56. Plus 1 coverage-probe metadata contact.
- **Period-boundary invalidation**: `invalid_cross_split` flags enforced at every period boundary; assertion logged in manifest.

## Expected Artifacts

When `RUN_V2_1=True`, the runner writes:

```text
results/v2_1_guarded_walkforward_readout/<run_id>/
  v2_1_walkforward_readout.csv
  v2_1_per_ticker_readout.csv
  v2_1_period_summary.csv
  v2_1_comparison_table.csv
  v2_1_same_row_baselines.csv
  v2_1_predictions.csv
  v2_1_decision_record.json
  v2_1_period_registry.json
  run_manifest.json
  artifact_inventory.csv
```

The durable-save cell uploads the run folder plus `drive_backup_manifest.json` to `My Drive/lst_models/results/v2_1_guarded_walkforward_readout/<run_id>/`. Per-period recovery checkpoints are mirrored to `My Drive/lst_models/checkpoints/v2_1_guarded_walkforward_readout/<run_id>/`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import hashlib
import shutil
import subprocess
import sys
import threading
import time
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "b901adfa38f6cdf55cf4a480119de37feee87cab"
PROJECT_ROOT = Path("/content/lst_models")

RUN_STAGE00_DRIVE_SYNC = True
RUN_STAGE01_DRIVE_SYNC = True
RUN_STAGE02_DRIVE_SYNC = True
RUN_STAGE03_DRIVE_SYNC = True
RUN_RAW_DATA_SYNC = True
RUN_V2_1_CHECKPOINT_MIRROR = True
RUN_DRIVE_BACKUP = True
RUN_V2_1 = False
RUN_ARTIFACT_DISPLAY = False

RESUME_V2_1_RUN_ID = ""
RESUME_V2_1_CHECKPOINT_DIR = ""

STAGE_NAME = "v2_1_guarded_walkforward_readout"
ROUTE = "lst_models"
SCOPE = "guarded_walkforward_readout"
HOLDOUT_TEST_CONTACT = True
HOLDOUT_CONTACT_TIER = "guarded_historically_contacted"
CLEAN_TEST_CLAIM = False
OFFICIAL_VALIDATION_SCORING_EVENTS = 0
NO_FINAL_MODEL_SELECTED = True
STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE02_RUN_ID = "20260610_082130_797479"
STAGE03_RUN_ID = "20260610_133305_716174"
STAGE04_RUN_ID = "20260610_232623_326133"
STAGE04_ORDERING = "stage04_first"
STAGE04_ORDERING_OVERRIDE_REASON = None
SUPERSEDED_STAGE02_RUN_IDS = ["20260609_100637_704705", "20260610_010019_507648"]
FROZEN_SEEDS = [101, 202]

STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE02_OUTPUT_DIR = Path("/content/lst_models_results/02_model_hpo_train_inner") / STAGE02_RUN_ID
STAGE03_OUTPUT_DIR = Path("/content/lst_models_results/03_frozen_validation_readout") / STAGE03_RUN_ID
OUTPUT_DIR = Path("/content/lst_models_results/v2_1_guarded_walkforward_readout")
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")
CHECKPOINT_ROOT = Path("/content/lst_models_checkpoints/v2_1_guarded_walkforward_readout")
STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE02_DRIVE_PATH_PARTS = ["lst_models", "results", "02_model_hpo_train_inner", STAGE02_RUN_ID]
STAGE03_DRIVE_PATH_PARTS = ["lst_models", "results", "03_frozen_validation_readout", STAGE03_RUN_ID]
V2_1_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_1_guarded_walkforward_readout"]
CHECKPOINT_DRIVE_BASE_PARTS = ["lst_models", "checkpoints", "v2_1_guarded_walkforward_readout"]

if STAGE02_RUN_ID in SUPERSEDED_STAGE02_RUN_IDS:
    raise ValueError(f"superseded Stage 02 run id must not be pinned: {STAGE02_RUN_ID}")

PERIODS = [
    {"period_id": "wf_p1", "start": "2017-01-25", "end_exclusive": "2018-01-25"},
    {"period_id": "wf_p2", "start": "2018-01-25", "end_exclusive": "2019-01-25"},
    {"period_id": "wf_p3", "start": "2019-01-25", "end_exclusive": "2020-01-25"},
    {"period_id": "wf_p4", "start": "2020-01-25", "end_exclusive": "2021-01-25"},
    {"period_id": "wf_p5", "start": "2021-01-25", "end_exclusive": "2022-01-25"},
    {"period_id": "wf_p6", "start": "2022-01-25", "end_exclusive": "2023-01-25"},
    {"period_id": "wf_p7", "start": "2023-01-25", "end_exclusive": "2024-04-19"},
]

MODEL_ROSTER = [
    {
        "table_row_id": "tcn_frozen_primary",
        "model_family": "tcn",
        "probe_id": "tcn_tiny",
        "hpo_profile_id": "tcn_p01",
        "params": {"channels": [16, 16], "kernel_size": 2, "dropout": 0.0,
                   "learning_rate": 0.001, "weight_decay": 0.0001},
    },
    {
        "table_row_id": "lightgbm_family_best",
        "model_family": "lightgbm",
        "probe_id": "lightgbm_small",
        "hpo_profile_id": "lgbm_p03",
        "params": {"n_estimators": 300, "learning_rate": 0.02, "max_depth": 8,
                   "num_leaves": 63, "min_child_samples": 100, "subsample": 0.9,
                   "colsample_bytree": 0.9, "reg_lambda": 5.0, "class_weight": "balanced"},
    },
    {
        "table_row_id": "standard_dlinear_family_best",
        "model_family": "standard_dlinear",
        "probe_id": "standard_dlinear_tiny",
        "hpo_profile_id": "dlinear_p03",
        "params": {"moving_avg_kernel": 7, "dropout": 0.10,
                   "learning_rate": 0.0003, "weight_decay": 0.0001},
    },
    {
        "table_row_id": "ms_dlinear_tcn_family_best",
        "model_family": "ms_dlinear_tcn",
        "probe_id": "ms_dlinear_tcn_tiny",
        "hpo_profile_id": "msdt_p01",
        "params": {"moving_avg_kernels": [3, 5, 9], "tcn_channels": [16, 16],
                   "tcn_kernel_size": 3, "dropout": 0.10,
                   "learning_rate": 0.001, "weight_decay": 0.0001},
    },
]

CANDIDATE_ID = "price_volume_time_w20"
FEATURE_SET = "price_volume_time"
WINDOW_SIZE = 20
LABEL_HORIZON_K = 9
LABEL_BAND_BPS = 3.0

V2_1_SIGN_OFF = {
    "status": "complete",
    "user_sign_off_date": "2026-06-17",
    "advisor_confirmation_reference": "gmail:19ebe45fd75d7f8b",
    "resolved_open_decisions": {
        "OD-A_period_count_k": len(PERIODS),
        "OD-B_period_length_months": 12,
        "OD-C_candidate_input_policy": "A_family_best_verbatim",
        "OD-D_ablation_rows_included": False,
        "OD-E_stage04_ordering": STAGE04_ORDERING,
        "OD-F_criteria_accepted": "yes",
    },
}
V2_1_COVERAGE_PROBE = {
    "authorization": "approved_after_sign_off",
    "artifact": "v2_1_coverage_probe.json",
    "artifact_sha256": "890a44d5455d4dfe14127adad3004f68e0c068a740067bf97afb31535fb8bf00",
    "probe_timestamp_utc": "2026-06-17T07:54:29.717476+00:00",
}

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(a) for a in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "v2_1_guarded_walkforward_readout.yaml").exists()
        and (path / "docs" / "protocols" / "v2_1_guarded_walkforward_readout_protocol.md").exists()
        and (path / "notebooks" / "v2_1_guarded_walkforward_readout_colab.ipynb").exists()
        and (path / "src" / "lst_models" / "stages" / "guarded_walkforward_readout.py").exists()
        and (path / "src" / "lst_models" / "guarded_walkforward.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "<" in PROJECT_REPO_COMMIT:
            raise ValueError("fill PROJECT_REPO_COMMIT with the V2.1 full-bundle commit before bootstrapping")
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
        ).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_1_guarded_walkforward_readout.yaml"
PROTOCOL_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_1_guarded_walkforward_readout_protocol.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "v2_1_guarded_walkforward_readout_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "guarded_walkforward_readout.py"
DOMAIN_MODULE_PATH = PROJECT_ROOT / "src" / "lst_models" / "guarded_walkforward.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
REQUIRED_PROJECT_FILES = [STAGE_CONFIG_PATH, PROTOCOL_PATH, NOTEBOOK_PATH, STAGE_ENTRYPOINT_PATH, DOMAIN_MODULE_PATH, RAW_DATA_MANIFEST_PATH]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("V2.1 bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def clear_project_import_cache():
    cached = [name for name in sys.modules if name == "lst_models" or name.startswith("lst_models.")]
    for name in cached:
        del sys.modules[name]
    importlib.invalidate_caches()

clear_project_import_cache()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_BOOTSTRAP_MODE:", PROJECT_BOOTSTRAP_MODE)
print("PROJECT_COMMIT:", PROJECT_REPO_COMMIT)
print("STAGE_NAME:", STAGE_NAME)
print("SCOPE:", SCOPE)
print("HOLDOUT_TEST_CONTACT:", HOLDOUT_TEST_CONTACT)
print("HOLDOUT_CONTACT_TIER:", HOLDOUT_CONTACT_TIER)
print("STAGE00_RUN_ID:", STAGE00_RUN_ID)
print("STAGE01_RUN_ID:", STAGE01_RUN_ID)
print("STAGE02_RUN_ID:", STAGE02_RUN_ID)
print("STAGE03_RUN_ID:", STAGE03_RUN_ID)
print("STAGE04_RUN_ID:", STAGE04_RUN_ID)
print("SUPERSEDED_STAGE02_RUN_IDS:", SUPERSEDED_STAGE02_RUN_IDS)
print("FROZEN_SEEDS:", FROZEN_SEEDS)
print("PERIODS:", len(PERIODS))
print("MODEL_ROSTER:", [m["table_row_id"] for m in MODEL_ROSTER])
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)
print("RUN_V2_1:", RUN_V2_1)
print("RUN_V2_1_CHECKPOINT_MIRROR:", RUN_V2_1_CHECKPOINT_MIRROR)
print("RUN_DRIVE_BACKUP:", RUN_DRIVE_BACKUP)

## Config Load, Runtime Injection, And Contract Check

This cell loads the V2.1 config sidecar, injects the notebook runtime paths and exact upstream run ids BEFORE the contract asserts (runtime paths are part of the executable contract), and checks the frozen blocks. It does not read data or fit anything.

In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the V2.1 config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required V2.1 file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PROTOCOL_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    v2_1_config = yaml.safe_load(handle)

stage_inputs = v2_1_config["inputs"]
stage_outputs = v2_1_config["outputs"]
stage_checkpointing = v2_1_config["checkpointing"]
stage_inputs["stage00_run_id"] = STAGE00_RUN_ID
stage_inputs["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
stage_inputs["stage00_drive_path_parts"] = STAGE00_DRIVE_PATH_PARTS
stage_inputs["stage01_run_id"] = STAGE01_RUN_ID
stage_inputs["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
stage_inputs["stage01_drive_path_parts"] = STAGE01_DRIVE_PATH_PARTS
stage_inputs["stage02_run_id"] = STAGE02_RUN_ID
stage_inputs["stage02_runtime_run_dir"] = str(STAGE02_OUTPUT_DIR)
stage_inputs["stage02_drive_path_parts"] = STAGE02_DRIVE_PATH_PARTS
stage_inputs["stage03_run_id"] = STAGE03_RUN_ID
stage_inputs["stage03_runtime_run_dir"] = str(STAGE03_OUTPUT_DIR)
stage_inputs["stage03_drive_path_parts"] = STAGE03_DRIVE_PATH_PARTS
stage_inputs["superseded_stage02_run_ids"] = SUPERSEDED_STAGE02_RUN_IDS
stage_inputs["raw_data_dir"] = str(RAW_DATA_DIR)
stage_inputs["stage04_run_id"] = STAGE04_RUN_ID
stage_inputs["stage04_ordering"] = STAGE04_ORDERING
stage_inputs["stage04_ordering_override_reason"] = STAGE04_ORDERING_OVERRIDE_REASON
stage_outputs["output_dir"] = str(OUTPUT_DIR)
stage_checkpointing["checkpoint_dir"] = str(CHECKPOINT_ROOT)
v2_1_config["sign_off"] = V2_1_SIGN_OFF
v2_1_config["coverage_probe"] = V2_1_COVERAGE_PROBE
if RESUME_V2_1_RUN_ID:
    resume_checkpoint_dir = RESUME_V2_1_CHECKPOINT_DIR or str(CHECKPOINT_ROOT / RESUME_V2_1_RUN_ID)
    v2_1_config["resume"] = {
        "enabled": True,
        "run_id": RESUME_V2_1_RUN_ID,
        "checkpoint_dir": resume_checkpoint_dir,
    }
    print("RESUME MODE: continuing run", RESUME_V2_1_RUN_ID, "from", resume_checkpoint_dir)

assert v2_1_config["stage_name"] == STAGE_NAME
assert v2_1_config["route"] == ROUTE
assert v2_1_config["scope"] == SCOPE
assert v2_1_config["holdout_test_contact"] is HOLDOUT_TEST_CONTACT
assert v2_1_config["holdout_contact_tier"] == HOLDOUT_CONTACT_TIER
assert v2_1_config["clean_test_claim"] is CLEAN_TEST_CLAIM
assert int(v2_1_config["official_validation_scoring_events"]) == OFFICIAL_VALIDATION_SCORING_EVENTS
assert v2_1_config["no_final_model_selected"] is NO_FINAL_MODEL_SELECTED
assert v2_1_config["v2_frozen_selection_unchanged"] is True
assert stage_inputs["stage00_run_id"] == STAGE00_RUN_ID
assert stage_inputs["stage01_run_id"] == STAGE01_RUN_ID
assert stage_inputs["stage02_run_id"] == STAGE02_RUN_ID
assert stage_inputs["stage03_run_id"] == STAGE03_RUN_ID
assert stage_inputs["stage04_run_id"] == STAGE04_RUN_ID
assert stage_inputs["stage04_ordering"] == STAGE04_ORDERING
assert stage_inputs["stage02_run_id"] not in stage_inputs["superseded_stage02_run_ids"]
assert Path(stage_inputs["raw_data_dir"]) == RAW_DATA_DIR
assert Path(stage_outputs["output_dir"]) == OUTPUT_DIR
assert Path(stage_checkpointing["checkpoint_dir"]) == CHECKPOINT_ROOT
assert v2_1_config["sign_off"]["status"] in {"pending", "complete"}
assert v2_1_config["coverage_probe"]["artifact"] == "v2_1_coverage_probe.json"

wf = v2_1_config["walkforward"]
assert wf["seeds"] == FROZEN_SEEDS
assert wf["period_count"] == len(PERIODS)
assert wf["score_each_period_model_seed_exactly_once"] is True
for i, p in enumerate(wf["periods"]):
    assert p["period_id"] == PERIODS[i]["period_id"]
    assert p["start"] == PERIODS[i]["start"]
    assert p["end_exclusive"] == PERIODS[i]["end_exclusive"]

roster = v2_1_config["model_roster"]
config_rows = {r["table_row_id"]: r for r in roster["model_rows"]}
for model in MODEL_ROSTER:
    row = config_rows[model["table_row_id"]]
    assert row["model_family"] == model["model_family"]
    assert row["probe_id"] == model["probe_id"]
    assert row["hpo_profile_id"] == model["hpo_profile_id"]

lgbm_defaults = v2_1_config["lightgbm_training_defaults"]
assert lgbm_defaults["early_stopping_validation_source"] == "inner_train_chronological_tail"
assert int(lgbm_defaults["early_stopping_rounds"]) == 25

torch_defaults = v2_1_config["probe_training_defaults"]["torch"]
assert torch_defaults["early_stopping"] == "inner_train_chronological_tail"
assert int(torch_defaults["early_stopping_patience"]) == 8
assert int(torch_defaults["epochs"]) == 64

assert v2_1_config["forbidden"]["wording"], "forbidden wording list must be non-empty"
assert "clean test" in v2_1_config["forbidden"]["wording"]

expected_output_names = {
    "manifest": "run_manifest.json",
    "artifact_inventory": "artifact_inventory.csv",
    "walkforward_readout": "v2_1_walkforward_readout.csv",
    "per_ticker_readout": "v2_1_per_ticker_readout.csv",
    "period_summary": "v2_1_period_summary.csv",
    "comparison_table": "v2_1_comparison_table.csv",
    "same_row_baselines": "v2_1_same_row_baselines.csv",
    "predictions": "v2_1_predictions.csv",
    "decision_record": "v2_1_decision_record.json",
    "period_registry": "v2_1_period_registry.json",
}
for output_key, output_name in expected_output_names.items():
    assert stage_outputs[output_key] == output_name

print(json.dumps({
    "stage_name": v2_1_config["stage_name"],
    "scope": v2_1_config["scope"],
    "holdout_contact_tier": v2_1_config["holdout_contact_tier"],
    "official_validation_scoring_events": v2_1_config["official_validation_scoring_events"],
    "period_count": wf["period_count"],
    "model_rows": [r["table_row_id"] for r in roster["model_rows"]],
    "holdout_test_contact": v2_1_config["holdout_test_contact"],
}, indent=2))

## Upstream Inputs: Stage 00-03 Run Folders And Raw Data

V2.1 verifies the full frozen chain before anything runs. If the Colab runtime was restarted, this cell downloads the four exact upstream run folders by Drive path parts (duplicate Drive matches are a hard error) and the raw ticker files by Drive file ID. Validation config (`_validate_config`) is checked before any Drive sync.

In [ ]:
from lst_models.artifacts import require_artifacts

def get_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive access only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive", fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def ensure_drive_folder(service, parent_id, name):
    folder_mime = "application/vnd.google-apps.folder"
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false and mimeType = '{folder_mime}'"
    matches = service.files().list(q=query, fields="files(id, name, webViewLink)", pageSize=10).execute().get("files", [])
    if len(matches) == 1:
        return matches[0]["id"]
    if len(matches) > 1:
        raise RuntimeError(f"Duplicate Drive folders named {name!r} under parent {parent_id}")
    created = service.files().create(body={"name": name, "mimeType": folder_mime, "parents": [parent_id]}, fields="id, name, webViewLink").execute()
    return created["id"]

def ensure_drive_path(service, path_parts):
    folder_id = "root"
    for part in path_parts:
        folder_id = ensure_drive_folder(service, folder_id, part)
    return folder_id

def find_drive_children(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false"
    return service.files().list(q=query, fields="files(id, name, mimeType, size, webViewLink)", pageSize=10).execute().get("files", [])

def upload_or_update_drive_file(service, parent_id, local_path, display_name=None):
    from googleapiclient.http import MediaFileUpload
    name = display_name or local_path.name
    matches = find_drive_children(service, parent_id, name)
    media = MediaFileUpload(str(local_path), resumable=True)
    if len(matches) == 0:
        uploaded = service.files().create(body={"name": name, "parents": [parent_id]}, media_body=media, fields="id, name, size, webViewLink").execute()
        action = "uploaded"
    elif len(matches) == 1:
        uploaded = service.files().update(fileId=matches[0]["id"], media_body=media, fields="id, name, size, webViewLink").execute()
        action = "updated"
    else:
        raise RuntimeError(f"Duplicate Drive files named {name!r} under parent {parent_id}")
    print(f"{action}: {name}")
    return dict(uploaded)

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_manifest["raw_source"]["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

required_stage00_artifacts = stage_inputs["required_stage00_artifacts"]
required_stage01_artifacts = stage_inputs["required_stage01_artifacts"]
required_stage02_artifacts = stage_inputs["required_stage02_artifacts"]
required_stage03_artifacts = stage_inputs["required_stage03_artifacts"]

if RUN_V2_1:
    from lst_models.stages.guarded_walkforward_readout import _validate_config
    _validate_config(v2_1_config)

    service = get_drive_service()
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    missing_raw = [raw_manifest["raw_source"]["files"][ticker]["name"] for ticker in raw_manifest["tickers"] if not (RAW_DATA_DIR / raw_manifest["raw_source"]["files"][ticker]["name"]).exists()]
    if missing_raw and RUN_RAW_DATA_SYNC:
        print("Missing raw data files; syncing from Drive file IDs:", missing_raw)
        sync_raw_data_from_drive(service)
    stage00_missing = [name for name in required_stage00_artifacts if not (STAGE00_OUTPUT_DIR / name).exists()]
    if stage00_missing and RUN_STAGE00_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS, stage00_missing, "stage00")
    stage01_missing = [name for name in required_stage01_artifacts if not (STAGE01_OUTPUT_DIR / name).exists()]
    if stage01_missing and RUN_STAGE01_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS, stage01_missing, "stage01")
    stage02_missing = [name for name in required_stage02_artifacts if not (STAGE02_OUTPUT_DIR / name).exists()]
    if stage02_missing and RUN_STAGE02_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE02_OUTPUT_DIR, STAGE02_DRIVE_PATH_PARTS, stage02_missing, "stage02")
    stage03_missing = [name for name in required_stage03_artifacts if not (STAGE03_OUTPUT_DIR / name).exists()]
    if stage03_missing and RUN_STAGE03_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE03_OUTPUT_DIR, STAGE03_DRIVE_PATH_PARTS, stage03_missing, "stage03")
    require_artifacts(STAGE00_OUTPUT_DIR, required_stage00_artifacts)
    require_artifacts(STAGE01_OUTPUT_DIR, required_stage01_artifacts)
    require_artifacts(STAGE02_OUTPUT_DIR, required_stage02_artifacts)
    require_artifacts(STAGE03_OUTPUT_DIR, required_stage03_artifacts)
    print("Stage 00-03 and raw data input checks passed.")
else:
    print("RUN_V2_1=False; upstream input sync skipped.")

## Checkpoint Mirror To Drive

The runner writes per-period recovery checkpoints (partial readout ledger plus `checkpoint_manifest.json`) under the local checkpoint root. This cell defines a background mirror that uploads mtime-stable checkpoint files to `My Drive/lst_models/checkpoints/v2_1_guarded_walkforward_readout/<run_id>/` during the run. Checkpoints are recovery state only, never evidence artifacts.

In [ ]:
V2_1_MIRROR_STATE = {"stop": False, "errors": [], "uploaded_mtimes": {}, "folder_ids": {}}
V2_1_MIRROR_POLL_SECONDS = 120
V2_1_MIRROR_STABLE_SECONDS = 10

def mirror_v2_1_checkpoints_once(service):
    if not CHECKPOINT_ROOT.exists():
        return 0
    uploaded_count = 0
    now = time.time()
    for path in sorted(CHECKPOINT_ROOT.rglob("*")):
        if not path.is_file():
            continue
        mtime = path.stat().st_mtime
        if now - mtime < V2_1_MIRROR_STABLE_SECONDS:
            continue
        key = str(path)
        if V2_1_MIRROR_STATE["uploaded_mtimes"].get(key) == mtime:
            continue
        relative = path.relative_to(CHECKPOINT_ROOT)
        parent_parts = tuple(CHECKPOINT_DRIVE_BASE_PARTS + list(relative.parent.parts)) if relative.parent != Path(".") else tuple(CHECKPOINT_DRIVE_BASE_PARTS)
        folder_id = V2_1_MIRROR_STATE["folder_ids"].get(parent_parts)
        if folder_id is None:
            folder_id = ensure_drive_path(service, list(parent_parts))
            V2_1_MIRROR_STATE["folder_ids"][parent_parts] = folder_id
        upload_or_update_drive_file(service, folder_id, path)
        V2_1_MIRROR_STATE["uploaded_mtimes"][key] = mtime
        uploaded_count += 1
    return uploaded_count

def v2_1_checkpoint_mirror_loop():
    try:
        mirror_service = get_drive_service()
    except Exception as exc:
        V2_1_MIRROR_STATE["errors"].append(f"mirror auth failed: {type(exc).__name__}: {exc}")
        print("checkpoint mirror disabled:", V2_1_MIRROR_STATE["errors"][-1])
        return
    while not V2_1_MIRROR_STATE["stop"]:
        try:
            mirror_v2_1_checkpoints_once(mirror_service)
        except Exception as exc:
            V2_1_MIRROR_STATE["errors"].append(f"{type(exc).__name__}: {exc}")
            print("checkpoint mirror error (recorded, run continues):", V2_1_MIRROR_STATE["errors"][-1])
        for _ in range(V2_1_MIRROR_POLL_SECONDS):
            if V2_1_MIRROR_STATE["stop"]:
                break
            time.sleep(1)

def start_v2_1_checkpoint_mirror():
    V2_1_MIRROR_STATE["stop"] = False
    thread = threading.Thread(target=v2_1_checkpoint_mirror_loop, name="v2_1_checkpoint_mirror", daemon=True)
    thread.start()
    return thread

def stop_v2_1_checkpoint_mirror(thread):
    V2_1_MIRROR_STATE["stop"] = True
    if thread is not None:
        thread.join(timeout=V2_1_MIRROR_POLL_SECONDS + 30)
    final_service = get_drive_service()
    V2_1_MIRROR_STATE["uploaded_mtimes"] = {}
    uploaded = mirror_v2_1_checkpoints_once(final_service)
    print(f"final checkpoint sweep uploaded {uploaded} files; recorded mirror errors: {len(V2_1_MIRROR_STATE['errors'])}")
    if V2_1_MIRROR_STATE["errors"]:
        print(json.dumps(V2_1_MIRROR_STATE["errors"], indent=2))

from lst_models.stages.guarded_walkforward_readout import RESUME_CHECKPOINT_FILES

RESUME_REQUIRED_CHECKPOINT_FILES = list(RESUME_CHECKPOINT_FILES)

def fetch_resume_checkpoint_from_drive():
    local_dir = Path(RESUME_V2_1_CHECKPOINT_DIR or str(CHECKPOINT_ROOT / RESUME_V2_1_RUN_ID))
    missing = [name for name in RESUME_REQUIRED_CHECKPOINT_FILES if not (local_dir / name).exists()]
    if not missing:
        print("resume checkpoint complete locally:", local_dir)
        return
    resume_service = get_drive_service()
    drive_parts = CHECKPOINT_DRIVE_BASE_PARTS + [RESUME_V2_1_RUN_ID]
    sync_stage_artifacts_from_drive(resume_service, local_dir, drive_parts, missing, "v2_1 resume checkpoint")
    still_missing = [name for name in RESUME_REQUIRED_CHECKPOINT_FILES if not (local_dir / name).exists()]
    if still_missing:
        raise FileNotFoundError(f"resume checkpoint files missing after Drive fetch: {still_missing}")

if RUN_V2_1 and RESUME_V2_1_RUN_ID:
    fetch_resume_checkpoint_from_drive()

print("Checkpoint mirror helpers ready; RUN_V2_1_CHECKPOINT_MIRROR =", RUN_V2_1_CHECKPOINT_MIRROR)

## Execute Walk-Forward Readout (via `run_stage`)

The runner module `src/lst_models/stages/guarded_walkforward_readout.py` handles data loading, walk-forward loop, refit, scoring, comparison table, stability judgement, and all artifact writes. The notebook calls `run_stage(config)` and receives a `V21Result` dataclass.

In [ ]:
if RUN_V2_1:
    try:
        from lst_models.stages.guarded_walkforward_readout import run_stage
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError("Missing V2.1 package entry point: src/lst_models/stages/guarded_walkforward_readout.py.") from exc

    mirror_thread = start_v2_1_checkpoint_mirror() if RUN_V2_1_CHECKPOINT_MIRROR else None
    try:
        result = run_stage(v2_1_config)
    finally:
        if RUN_V2_1_CHECKPOINT_MIRROR:
            stop_v2_1_checkpoint_mirror(mirror_thread)

    run_id = result.output_dir.name
    run_dir = result.output_dir

    if run_dir.parent != OUTPUT_DIR:
        raise RuntimeError(f"V2.1 output dir mismatch: {run_dir.parent} != {OUTPUT_DIR}")

    print(f"RUN_ID: {run_id}")
    print(f"RUN_DIR: {run_dir}")
    print(f"Manifest: {result.run_manifest}")
    print(f"Decision: {result.decision_record}")
    print()
    for f in sorted(run_dir.iterdir()):
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")
else:
    print("RUN_V2_1=False; walk-forward readout skipped.")

## Durable Drive Result Save

Immediately after `run_stage` returns, this cell validates the V2.1 required outputs, refuses upload unless the run manifest records `holdout_contact_tier=guarded_historically_contacted`, `clean_test_claim=false`, `no_final_model_selected=true`, and `official_validation_scoring_events=0`, and uploads the run folder to `My Drive/lst_models/results/v2_1_guarded_walkforward_readout/<run_id>/`. `drive_backup_manifest.json` is written and uploaded last; its own entry records size null (self-reference).

In [ ]:
from lst_models.stages.guarded_walkforward_readout import REQUIRED_V2_1_ARTIFACTS

if RUN_DRIVE_BACKUP and RUN_V2_1:
    if "result" not in globals():
        raise RuntimeError("RUN_DRIVE_BACKUP=True requires running V2.1 first.")

    missing_required_artifacts = [
        name for name in REQUIRED_V2_1_ARTIFACTS if not (run_dir / name).exists()
    ]
    if missing_required_artifacts:
        missing_text = ", ".join(missing_required_artifacts)
        raise FileNotFoundError(
            "Missing required V2.1 artifacts before Drive backup in "
            f"{run_dir}: {missing_text}"
        )

    manifest_data = json.loads(result.run_manifest.read_text(encoding="utf-8"))
    decision_data = json.loads(result.decision_record.read_text(encoding="utf-8"))

    if manifest_data.get("holdout_contact_tier") != "guarded_historically_contacted":
        raise ValueError("backup refused: run manifest must record holdout_contact_tier=guarded_historically_contacted")
    if manifest_data.get("clean_test_claim") is not False:
        raise ValueError("backup refused: run manifest must record clean_test_claim=false")
    if manifest_data.get("no_final_model_selected") is not True:
        raise ValueError("backup refused: run manifest must record no_final_model_selected=true")
    if int(manifest_data.get("official_validation_scoring_events", -1)) != 0:
        raise ValueError("backup refused: run manifest must record official_validation_scoring_events=0")

    drive_path_parts = V2_1_DRIVE_RESULT_PATH_PARTS + [run_id]
    drive_folder_id = ensure_drive_path(service, drive_path_parts)

    backup_manifest_path = run_dir / "drive_backup_manifest.json"
    local_files = sorted(path for path in run_dir.rglob("*") if path.is_file() and path.name != backup_manifest_path.name)
    uploads = []
    for path in local_files:
        uploaded = upload_or_update_drive_file(service, drive_folder_id, path)
        uploaded["bytes"] = path.stat().st_size
        uploads.append(uploaded)

    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": run_id,
        "stage_run_id": run_id,
        "decision": decision_data["decision"],
        "guarded_scoring_events": decision_data["guarded_scoring_events"],
        "local_output_dir": str(run_dir),
        "drive_path": "My Drive/" + "/".join(drive_path_parts),
        "drive_path_parts": drive_path_parts,
        "drive_folder_id": drive_folder_id,
        "uploaded_file_names": [upload["name"] for upload in uploads],
        "uploaded_files": uploads,
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "holdout_contact_tier": "guarded_historically_contacted",
        "clean_test_claim": False,
        "no_final_model_selected": True,
        "official_validation_scoring_events": 0,
    }
    backup_manifest["uploaded_files"] = backup_manifest["uploaded_files"] + [
        {"name": backup_manifest_path.name, "bytes": None, "self_reference": True}
    ]
    backup_manifest["uploaded_file_names"].append(backup_manifest_path.name)
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)

    print("stage_run_id:", backup_manifest["stage_run_id"])
    print("drive_path:", backup_manifest["drive_path"])
    print("drive_folder_id:", backup_manifest["drive_folder_id"])
else:
    print("RUN_DRIVE_BACKUP is disabled or RUN_V2_1=False; no V2.1 result backup uploaded.")

## Walk-Forward Results: Comparison Table And Period Summary

In [ ]:
if RUN_V2_1:
    import pandas as pd

    print("=== Ian's Comparison Table (Guarded Walk-Forward Readout) ===")
    comparison_df = pd.read_csv(result.comparison_table)
    display(comparison_df)

    print("\n=== Period Summary (mean over seeds, per period x model) ===")
    period_df = pd.read_csv(result.period_summary)
    display(period_df)
else:
    print("RUN_V2_1=False; results display skipped.")

## Stability Criteria And Decision Record

In [ ]:
if RUN_V2_1:
    decision_data = json.loads(result.decision_record.read_text(encoding="utf-8"))

    print("=== Stability Criteria (tcn_frozen_primary) ===")
    print(f"  positive_period_count = {decision_data['positive_period_count']}"
          f" (>= 2: {decision_data['criteria_met']['positive_period_count']})")
    print(f"  pooled_delta = {decision_data['pooled_delta']:.6f}"
          f" (> 0: {decision_data['criteria_met']['pooled_delta']})")
    print(f"  DECISION: {decision_data['decision']}")
    print(f"  readout_complete: {decision_data['readout_complete']}")
    print(f"  guarded_scoring_events: {decision_data['guarded_scoring_events']}")
else:
    print("RUN_V2_1=False; decision display skipped.")

## Readout Display

Allowed wording: `guarded walk-forward evidence`, `guarded period-stability readout`, `primary candidate met/did not meet predeclared guarded stability criteria`. Do not claim a final model, clean test, or out-of-sample proof from this notebook.

In [ ]:
if RUN_ARTIFACT_DISPLAY and RUN_V2_1:
    import pandas as pd

    decision_data = json.loads(result.decision_record.read_text(encoding="utf-8"))
    print("=== Decision Record Summary ===")
    print(json.dumps({
        "decision": decision_data["decision"],
        "positive_period_count": decision_data["positive_period_count"],
        "pooled_delta": decision_data["pooled_delta"],
        "criteria_met": decision_data["criteria_met"],
        "guarded_scoring_events": decision_data["guarded_scoring_events"],
        "readout_complete": decision_data["readout_complete"],
    }, indent=2))

    print("\n=== Comparison Table ===")
    display(pd.read_csv(result.comparison_table))

    print("\n=== Period Summary ===")
    display(pd.read_csv(result.period_summary))

    print("\n=== Full Readout (first 20 rows) ===")
    display(pd.read_csv(result.walkforward_readout).head(20))

    print("\n=== Baselines ===")
    display(pd.read_csv(result.same_row_baselines))
else:
    print("RUN_ARTIFACT_DISPLAY=False or RUN_V2_1=False; display skipped.")